# K-Nearest Neighbors (KNN) — Grupo 32

Este notebook apresenta os experimentos realizados com o algoritmo KNN para classificar os estados do Jogo da Velha.

Foram avaliadas duas abordagens de representação dos tabuleiros e diferentes valores de K, utilizando o F1 macro para selecionar a melhor configuração.

## 1. Configuração dos experimentos

O algoritmo KNN foi avaliado utilizando duas abordagens de representação dos tabuleiros.

Foram testados os valores de K iguais a 1, 3, 5, 7, 9, 11, 15 e 21.

A seleção da melhor configuração foi realizada com base no F1 macro obtido no conjunto de validação.

In [14]:
from pathlib import Path

import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


In [28]:
RAIZ_32 = Path.cwd().parents[1]
print(RAIZ_32)

VALORES_K_32 = [1, 3, 5, 7, 9, 11, 15, 21]

CLASSES_32 = [
    "Tem jogo",
    "X venceu",
    "O venceu",
    "Empate"
]

/Users/jesalvatori/Documents/PUCRS/tic-tac-toe-game


## 2. Carregamento e preparação dos dados

Os dados estão divididos em treinamento, validação e teste.

A Abordagem 1 representa as nove posições do tabuleiro, enquanto a Abordagem 2 utiliza 15 características derivadas.

O KNN utiliza a padronização dos atributos com `StandardScaler` antes de realizar a classificação.

O conjunto de treinamento utilizado nos experimentos possui 560 registros após o balanceamento das classes. Os conjuntos de validação e teste possuem 93 registros cada. O balanceamento foi aplicado apenas aos dados de treinamento, preservando a distribuição original dos demais conjuntos.


In [29]:
def carregar_dados_32(abordagem):
    pasta = RAIZ_32 / "dataset" / abordagem

    treino = pd.read_csv(pasta / "treino.csv")
    validacao = pd.read_csv(pasta / "validacao.csv")
    teste = pd.read_csv(pasta / "teste.csv")

    return treino, validacao, teste


def separar_dados_32(dados):
    X = dados.drop(columns=["classe"])
    y = dados["classe"]
    return X, y


def criar_modelo_32(k):
    return Pipeline([
        ("normalizacao", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])

## 3. Treinamento e validação do KNN

Foram testados diferentes valores de K nas duas abordagens de representação dos tabuleiros.

Cada modelo foi treinado com os dados de treinamento e avaliado no conjunto de validação. A configuração com maior F1 macro foi selecionada para a avaliação final no conjunto de teste.

Na avaliação final, foram calculadas a acurácia, a precisão macro, o recall macro, o F1 macro e a matriz de confusão.

In [30]:
def executar_32():
    melhor_f1_32 = -1
    melhor_k_32 = None
    melhor_abordagem_32 = None
    melhor_modelo_32 = None

    for abordagem_32 in ["abordagem_1", "abordagem_2"]:
        treino, validacao, _ = carregar_dados_32(abordagem_32)

        X_treino, y_treino = separar_dados_32(treino)
        X_validacao, y_validacao = separar_dados_32(validacao)

        print(f"\n{abordagem_32.upper()}")

        for k_32 in VALORES_K_32:
            modelo_32 = criar_modelo_32(k_32)
            modelo_32.fit(X_treino, y_treino)

            previsoes_32 = modelo_32.predict(X_validacao)

            f1_32 = f1_score(
                y_validacao,
                previsoes_32,
                average="macro",
                zero_division=0
            )

            print(f"K = {k_32} | F1 macro = {f1_32:.4f}")

            if f1_32 > melhor_f1_32:
                melhor_f1_32 = f1_32
                melhor_k_32 = k_32
                melhor_abordagem_32 = abordagem_32
                melhor_modelo_32 = modelo_32

    print("\nMELHOR CONFIGURACAO DO KNN")
    print(f"Abordagem: {melhor_abordagem_32}")
    print(f"K: {melhor_k_32}")
    print(f"F1 macro na validacao: {melhor_f1_32:.4f}")

    _, _, teste = carregar_dados_32(melhor_abordagem_32)
    X_teste, y_teste = separar_dados_32(teste)

    previsoes_teste_32 = melhor_modelo_32.predict(X_teste)

    acuracia_32 = accuracy_score(y_teste, previsoes_teste_32)
    precisao_32 = precision_score(
        y_teste, previsoes_teste_32,
        average="macro", zero_division=0
    )
    recall_32 = recall_score(
        y_teste, previsoes_teste_32,
        average="macro", zero_division=0
    )
    f1_teste_32 = f1_score(
        y_teste, previsoes_teste_32,
        average="macro", zero_division=0
    )

    print("\nRESULTADOS NO TESTE")
    print(f"Acuracia: {acuracia_32:.4f}")
    print(f"Precisao macro: {precisao_32:.4f}")
    print(f"Recall macro: {recall_32:.4f}")
    print(f"F1 macro: {f1_teste_32:.4f}")

    print("\nRELATORIO POR CLASSE")
    print(classification_report(
        y_teste,
        previsoes_teste_32,
        labels=CLASSES_32,
        zero_division=0
    ))

    print("\nMATRIZ DE CONFUSAO")
    print("Ordem das classes:", CLASSES_32)
    print(confusion_matrix(
        y_teste,
        previsoes_teste_32,
        labels=CLASSES_32
    ))

In [31]:
executar_32()


ABORDAGEM_1
K = 1 | F1 macro = 0.5370
K = 3 | F1 macro = 0.5603
K = 5 | F1 macro = 0.5140
K = 7 | F1 macro = 0.5507
K = 9 | F1 macro = 0.5285
K = 11 | F1 macro = 0.5220
K = 15 | F1 macro = 0.4819
K = 21 | F1 macro = 0.4413

ABORDAGEM_2
K = 1 | F1 macro = 0.7637
K = 3 | F1 macro = 0.7906
K = 5 | F1 macro = 0.7905
K = 7 | F1 macro = 0.8000
K = 9 | F1 macro = 0.8168
K = 11 | F1 macro = 0.7791
K = 15 | F1 macro = 0.7998
K = 21 | F1 macro = 0.7693

MELHOR CONFIGURACAO DO KNN
Abordagem: abordagem_2
K: 9
F1 macro na validacao: 0.8168

RESULTADOS NO TESTE
Acuracia: 0.8280
Precisao macro: 0.7981
Recall macro: 0.8667
F1 macro: 0.8050

RELATORIO POR CLASSE
              precision    recall  f1-score   support

    Tem jogo       0.94      0.57      0.71        30
    X venceu       0.71      0.90      0.79        30
    O venceu       0.94      1.00      0.97        30
      Empate       0.60      1.00      0.75         3

    accuracy                           0.83        93
   macro avg       

## 4. Resultados dos experimentos

### 4.1 Comparação entre as abordagens

O algoritmo KNN foi avaliado nas duas abordagens de representação dos tabuleiros, utilizando oito valores diferentes de K. O desempenho foi medido pelo F1 macro no conjunto de validação.

| Valor de K | Abordagem 1 | Abordagem 2 |
|---|---:|---:|
| 1 | 0,5370 | 0,7637 |
| 3 | 0,5603 | 0,7906 |
| 5 | 0,5140 | 0,7905 |
| 7 | 0,5507 | 0,8000 |
| 9 | 0,5285 | **0,8168** |
| 11 | 0,5220 | 0,7791 |
| 15 | 0,4819 | 0,7998 |
| 21 | 0,4413 | 0,7693 |

A Abordagem 2 apresentou resultados superiores em todos os valores de K testados.

Na Abordagem 1, o melhor resultado foi obtido com K = 3, alcançando F1 macro de 0,5603. Já na Abordagem 2, o melhor resultado foi obtido com K = 9, alcançando F1 macro de 0,8168.

Dessa forma, selecionamos a Abordagem 2 com K = 9 para a avaliação final no conjunto de teste.

### 4.2 Gráfico comparativo




